<center><img src="./image/dice.png" alt="dice" style="width: 300px;"/></center>
# 03.虚拟仿真机械臂物块抓取实验
<br>@－－－－dice－－－－<br>
@  文件名：03.虚拟仿真机械臂物块抓取实验.ipynb <br>
@  版本：V2.0 <br>
@  说明：Jetson Nano 虚拟仿真应用<br>
本实验属于机械臂基虚拟仿真实验部分<br>

## 导入依赖

In [1]:
! export DISPLAY=:0 ipython notebook

In [2]:
import time
import numpy as np
from numpy import pi
import pybullet
import pybullet_data
# 自定义库
from transform import Transform
from pose import Pose
from arm5dof_pybullet import Arm5DoFPyBullet
from arm_application import ArmApplication
# 设置Numpy的打印选项
# 精确位数3，不启用科学计数法
np.set_printoptions(precision=5, suppress=True)

pybullet build time: Jul  8 2022 13:29:02


## 创建机械臂工作场景

In [3]:
# 创建机械臂
# 传入机械臂路径
arm = Arm5DoFPyBullet("urdf/arm5dof.urdf")

startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Mesa/X.org
GL_RENDERER=llvmpipe (LLVM 12.0.0, 128 bits)
GL_VERSION=4.5 (Core Profile) Mesa 21.2.6
GL_SHADING_LANGUAGE_VERSION=4.50
pthread_getconcurrency()=0
Version = 4.5 (Core Profile) Mesa 21.2.6
Vendor = Mesa/X.org
Renderer = llvmpipe (LLVM 12.0.0, 128 bits)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
ven = Mesa/X.org
ven = Mesa/X.org
b3Printf: b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:

b3Printf: No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, ide

In [4]:
# 载入工作台0
# 1. 开启机械臂连杆的自碰撞检测, 不检测与父连杆的碰撞
flags = pybullet.URDF_USE_SELF_COLLISION_EXCLUDE_PARENT 
# 2. 使用URDF模型里面的转动惯量
flags |= pybullet.URDF_USE_INERTIA_FROM_FILE 
ws_board_id = pybullet.loadURDF("urdf/ws_board.urdf",\
                    [0.185, 0.0, 0.01], [0, 0, 0, 1], \
                        useFixedBase=True, flags=flags)



In [5]:
# 载入工作台1
ws_board2_id = pybullet.loadURDF("urdf/ws_board.urdf",\
                    [0, 0.185, 0.01], pybullet.getQuaternionFromEuler([0, 0, np.pi/2]), \
                        useFixedBase=True, flags=flags)

In [6]:
# 载入物块
cubic_posi = [0.2, 0, 0.1]
cubic_q = [0, 0, 0, 1]
# 开启图形形状的缓冲
# 将会缓存并复用图形几何形状，将会提高相同模型文件载入时的性能。
# 同时也降低对资源的占用。
flags = pybullet.URDF_ENABLE_CACHED_GRAPHICS_SHAPES
# 物块可以在空间中自由移动，所以useFixedBase=False
cubic_id = pybullet.loadURDF("urdf/cube.urdf", cubic_posi, cubic_q, \
                           useFixedBase=False, flags=flags)
print(f"物块id: {cubic_id}")
# 等待一会儿
time.sleep(3)

物块id: 4


## 物块在工作台坐标系下的位姿

In [7]:
# 获取工作台在世界坐标系下的位姿
ws_posi, ws_quat = pybullet.getBasePositionAndOrientation(ws_board_id)
ws_pose = Pose()
ws_pose.from_bullet_pose(ws_posi, ws_quat)
print("工作台在世界坐标系下的位姿")
print(ws_pose)

工作台在世界坐标系下的位姿
Pose x=185.0 mm, y=6.5 mm, z=10.0 mm, roll=0.0, pitch=-0.0, yaw=0.0


In [8]:
def get_cubic_posi(cubic_id):
    # 获取物块在工作台坐标系下的位置(PyBullet格式)
    cube_posi, cube_quat = pybullet.getBasePositionAndOrientation(cubic_id)
    cube_pose = Pose()
    cube_pose.from_bullet_pose(cube_posi, cube_quat)
    print("物块在世界坐标系下的位姿")
    print(cube_pose)

    # 获取物块在世界坐标系下的位姿
    T_world2ws = ws_pose.get_transform_matrix(unit="mm")
    T_world2cube = cube_pose.get_transform_matrix(unit="mm")
    T_ws2cube = np.dot(Transform.inverse(T_world2ws), T_world2cube)
    print(f"T_ws2cube: \n{T_ws2cube}")

    # 物块在工作台上的坐标
    t_ws2cube = T_ws2cube[:3, 3]
    print(f"t_ws2cube: \n{t_ws2cube}")
    return t_ws2cube

## 工作台在机械臂基坐标系下的位姿

In [9]:
def get_arm2ws():
    '''获取机械臂基坐标系到工作台坐标系的空间变换'''
    # 获取工作台在世界坐标系的位姿
    T_world2ws = ws_pose.get_transform_matrix(unit="mm")
    # 获取机械臂极坐标系位姿
    arm_base_posi, arm_base_quat = pybullet.getLinkState(arm.arm_id, 1)[:2]
    arm_base_pose = Pose()
    arm_base_pose.from_bullet_pose(arm_base_posi, arm_base_quat)
    print("机械臂基坐标系在世界坐标系下的位姿")
    print(arm_base_pose)
    
    T_world2arm = arm_base_pose.get_transform_matrix(unit="mm")
    T_arm2ws = np.dot(Transform.inverse(T_world2arm), T_world2ws)
    print(f"T_arm2ws: \n{T_arm2ws}")
    return T_arm2ws

In [10]:
T_arm2ws = get_arm2ws()

机械臂基坐标系在世界坐标系下的位姿
Pose x=-9.8 mm, y=0.0 mm, z=66.6 mm, roll=-0.0, pitch=6.9, yaw=0.1
T_arm2ws: 
[[  0.99279   0.00099  -0.11985 200.1882 ]
 [ -0.00103   1.       -0.00026   6.28572]
 [  0.11984   0.00039   0.99279 -32.82635]
 [  0.        0.        0.        1.     ]]


## 创建机械臂应用对象

In [11]:
arm_app = ArmApplication(arm, T_arm2ws)

In [12]:
# 配置放置区域
arm_app.GO_AWAY_FROM_WORKSPACE = [0, 185, 30]

In [13]:
# 获取物块位置
t_ws2cube = get_cubic_posi(cubic_id)
# 抓取
wx, wy, wz = t_ws2cube
arm_app.grab_cubic(wx, wy, wz)

物块在世界坐标系下的位姿
Pose x=200.0 mm, y=-0.0 mm, z=27.0 mm, roll=0.0, pitch=0.0, yaw=0.0
T_ws2cube: 
[[ 1.      -0.       0.00008 15.00168]
 [ 0.       1.      -0.      -6.47323]
 [-0.00008  0.       1.      16.99031]
 [ 0.       0.       0.       1.     ]]
t_ws2cube: 
[15.00168 -6.47323 16.99031]
